# Рекомендации на основе содержания

## Импорт данных из файла 

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

# Шаг 1: Загрузка данных
df_movies = pd.read_csv('ml-latest/movies.csv')
df_tags = pd.read_csv('ml-latest/tags.csv')
df_ratings = pd.read_csv('ml-latest/ratings.csv')

# Объединяем последовательно по нескольким столбцам
df = df_movies.merge(df_tags, on='movieId', how='inner').merge(df_ratings, on=['movieId', 'userId'], how='inner')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3476 entries, 0 to 3475
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   movieId      3476 non-null   int64  
 1   title        3476 non-null   object 
 2   genres       3476 non-null   object 
 3   userId       3476 non-null   int64  
 4   tag          3476 non-null   object 
 5   timestamp_x  3476 non-null   int64  
 6   rating       3476 non-null   float64
 7   timestamp_y  3476 non-null   int64  
dtypes: float64(1), int64(4), object(3)
memory usage: 217.4+ KB


## Преобразуем признаки жанра и тегов

In [8]:
#заменяем '|' на пробел в жанрах и объединяем с тегами
# Используем .fillna(''), чтобы избежать ошибок, если где-то есть пустые ячейки
genres_clean = df['genres'].fillna('').str.replace('|', ' ', regex=False)
tags_clean = df['tag'].fillna('')

# Создаем единое текстовое поле для TF-IDF
df['combined_features'] = genres_clean + ' ' + tags_clean

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df['combined_features'])

feature_names = vectorizer.get_feature_names_out()
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=feature_names)

# Объединяем с исходными ID пользователей и названиями фильмов
result_df = pd.concat([df[['movieId', 'userId', 'rating']], tfidf_df], axis=1)

# Выводим результат
print(result_df.head())

   movieId  userId  rating   06  1900s  1920s  1950s  1960s  1970s  1980s  \
0        1     336     4.0  0.0    0.0    0.0    0.0    0.0    0.0    0.0   
1        1     474     4.0  0.0    0.0    0.0    0.0    0.0    0.0    0.0   
2        1     567     3.5  0.0    0.0    0.0    0.0    0.0    0.0    0.0   
3        2      62     4.0  0.0    0.0    0.0    0.0    0.0    0.0    0.0   
4        2      62     4.0  0.0    0.0    0.0    0.0    0.0    0.0    0.0   

   ...  york  you  younger  your  zellweger  zither  zoe  zombie  zombies  \
0  ...   0.0  0.0      0.0   0.0        0.0     0.0  0.0     0.0      0.0   
1  ...   0.0  0.0      0.0   0.0        0.0     0.0  0.0     0.0      0.0   
2  ...   0.0  0.0      0.0   0.0        0.0     0.0  0.0     0.0      0.0   
3  ...   0.0  0.0      0.0   0.0        0.0     0.0  0.0     0.0      0.0   
4  ...   0.0  0.0      0.0   0.0        0.0     0.0  0.0     0.0      0.0   

   zooey  
0    0.0  
1    0.0  
2    0.0  
3    0.0  
4    0.0  

[5 rows

## Рассчет средних оценок пользователя и фильма

In [9]:
result_df["movie_mean_rating"] = (
    df.groupby("movieId")["rating"].transform("mean")
)

result_df["user_mean_rating"] = (
    df.groupby("userId")["rating"].transform("mean")
)

print(
    result_df[
        ["userId", "movieId", "rating",
         "movie_mean_rating", "user_mean_rating"]
    ].head(5).to_string(index=False)
)

 userId  movieId  rating  movie_mean_rating  user_mean_rating
    336        1     4.0           3.833333          3.777778
    474        1     4.0           3.833333          3.701909
    567        1     3.5           3.833333          3.917824
     62        2     4.0           3.750000          3.937838
     62        2     4.0           3.750000          3.937838


## Построение модели

In [12]:
# Разделение данных на train/test
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import Normalizer

# Нормализация и масштабирование
normalizer = Normalizer()
X_normalized = normalizer.fit_transform(result_df)

y = df['rating']
X_train, X_test, y_train, y_test = train_test_split(
    X_normalized,
    y,
    test_size=0.2,
    random_state=42
)

# Обучение модели
model = LinearRegression()
model.fit(X_train, y_train)

# Предсказание и оценка
y_pred = model.predict(X_test)

rmse = root_mean_squared_error(y_test, y_pred)
print(f"RMSE: {rmse}")

RMSE: 11.866076171359028
